## **LangChain** - _LangChain Expression Language_

Para entender la utilidad de **LangChain**, imaginemos que queremos crear una aplicación que resuma textos en diferentes idiomas. 

Si no usáramos **LangChain** y solo hiciéramos llamadas directas, tendríamos que usar la concatenación de strings de Python. Esto funciona para un script pequeño, pero cuando tienes prompts complejos, historial de chat y múltiples variables, el código se vuelve difícil de mantener y propenso a errores.

Veamos cómo **LangChain** resuelve esto usando **Plantillas de Prompt (PromptTemplates)** y su sintaxis de tuberías o *pipes* (`|`).

In [1]:
from dotenv import load_dotenv
from pprint import pprint

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

# Variables de entorno
load_dotenv()

MODEL = "gemini-3.1-flash-lite"
TEMPERATURE = 0

c:\Users\danie\OneDrive\Documentos\GitHub\DAIXXRT\LangChain - Boost Academy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 01. **PromptTemplate**

Un `PromptTemplate` (Plantilla de Prompt) es como **un formulario con espacios en blanco**. 

Su función principal es inyectar variables en una cadena de texto de forma segura y validada.

Usar `PromptTemplate` ofrece tres ventajas técnicas sobre el texto normal:

1. **Validación automática:** Si tu plantilla exige la variable `{idioma}` y olvidas pasarla, LangChain detiene el programa y te avisa inmediatamente con un error claro, antes de gastar dinero llamando a la API.
2. **Integración con LCEL:** Un string normal de Python no puede conectarse con el operador `|`. Un `PromptTemplate` es un objeto `Runnable` que sabe cómo recibir un diccionario de datos y enviárselo directamente al modelo.
3. **Serialización:** Puedes guardar tus prompts como archivos **JSON** o **YAML** y cargarlos en cualquier proyecto, separando el texto del código lógico.

In [2]:
# Inicializamos el cliente del modelo
llm = ChatGoogleGenerativeAI(model = MODEL, temperature = TEMPERATURE)

# Definimos una plantilla con variables dinámicas entre llaves {}
plantilla_texto = """
Eres un asistente experto en redacción. 
Tu tarea es resumir el siguiente texto en exactamente 2 oraciones, y luego traducirlo al {idioma}.

Texto original: {texto}

Resumen y traducción:
"""

# Creamos el objeto PromptTemplate de LangChain
prompt = PromptTemplate(input_variables = ["idioma", "texto"],
                        template = plantilla_texto)

# Probemos qué hace el prompt por sí solo (solo formatea el texto)
prompt_formateado = prompt.format(idioma = "inglés",
                                  texto = "LangChain es un framework de código abierto diseñado para facilitar la creación de aplicaciones utilizando grandes modelos de lenguaje (LLMs). Ofrece un conjunto de herramientas, componentes y abstracciones que simplifican la integración de estos modelos en diversas tareas.")

print("Visualizacion del Prompt:")
print(prompt_formateado)

Visualizacion del Prompt:

Eres un asistente experto en redacción. 
Tu tarea es resumir el siguiente texto en exactamente 2 oraciones, y luego traducirlo al inglés.

Texto original: LangChain es un framework de código abierto diseñado para facilitar la creación de aplicaciones utilizando grandes modelos de lenguaje (LLMs). Ofrece un conjunto de herramientas, componentes y abstracciones que simplifican la integración de estos modelos en diversas tareas.

Resumen y traducción:



### 02. _LangChain Expression Language_ (**LCEL**)

**LCEL** es la sintaxis declarativa que utiliza el operador `|` (pipe) para componer cadenas de ejecución (Runnables).**La salida del componente izquierdo se convierte en la entrada del componente derecho.**

```python
Cadena = Prompt | Modelo | Parser
```

¿Por qué usar **LCEL** en lugar de código Python normal?

1. **Soporte nativo de Streaming:** Las cadenas **LCEL** saben cómo enviar la respuesta letra por letra automáticamente si usas `.stream()`.
2. **Depuración fácil:** Puedes inspeccionar o interceptar los datos en cualquier punto del pipeline.

In [5]:
from langchain_core.output_parsers import StrOutputParser

# Instanciamos un "parser" que limpia la respuesta del modelo y nos devuelve solo el texto
parser = StrOutputParser()

# Creamos la cadena conectando los componentes usando el operador pipe
cadena_resumen = prompt | llm | parser

# Ejecutamos la cadena pasando un diccionario con nuestras variables

idioma = "deutch"

texto_largo = """
La inteligencia artificial generativa es una rama de la IA que se enfoca en crear contenido original, 
como texto, imágenes, audio o código, a partir de patrones aprendidos en grandes cantidades de datos. 
A diferencia de la IA tradicional, que suele clasificar o predecir basándose en datos existentes, 
la IA generativa utiliza modelos complejos, como redes neuronales profundas, para generar 
resultados nuevos y creativos que imitan la producción humana.
"""

llm_input = {"idioma" : idioma,
             "texto" : texto_largo}

# Usamos .invoke() enviando los datos crudos. La cadena hace el resto.
resultado = cadena_resumen.invoke(input = llm_input)

print(resultado)

Aquí tienes el resumen solicitado y su traducción al alemán:

**Resumen:**
La inteligencia artificial generativa utiliza modelos complejos para crear contenido original, como texto o imágenes, basándose en patrones aprendidos. A diferencia de la IA tradicional, esta tecnología genera resultados nuevos y creativos que imitan la producción humana.

**Traducción al alemán:**
Generative künstliche Intelligenz nutzt komplexe Modelle, um auf Basis erlernter Muster originäre Inhalte wie Texte oder Bilder zu erstellen. Im Gegensatz zur traditionellen KI erzeugt diese Technologie neue und kreative Ergebnisse, die menschliche Leistungen nachahmen.


### 03. **ChatPromptTemplate**

Hoy en día, modelos como **Gemini 3.1** o **GPT-4** están entrenados para conversar. No esperan un bloque de texto gigante, sino una **lista estructurada de mensajes** donde cada mensaje tiene un "Rol".

Los roles principales son:
* **System (Sistema):** Le da el contexto, personalidad o instrucciones generales al modelo (ej. "Eres un asistente útil y respondes en JSON"). Generalmente va primero.
* **Human (Humano / Usuario):** La pregunta o instrucción del usuario real.
* **AI (Asistente):** Las respuestas previas del modelo. Se usa para darle "memoria" a la conversación.

**LangChain** maneja esto con `ChatPromptTemplate`.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

# Creamos una plantilla estructurada por roles usando una lista de tuplas
prompt_chat = ChatPromptTemplate.from_messages(messages = [("system", "Eres un experto en {lenguaje_programacion}. Tu tono es sarcástico pero educativo."),
                                                           ("human", "¿Por qué mi código lanza un error de tipo {tipo_error}?"),
                                                           ("ai", "Oh, claro, el clásico error de novato. Déjame adivinar..."),
                                                           ("human", "¡Oye! Solo dime cómo solucionarlo, por favor.")])

# Usamos .format_messages() para ver la estructura real de objetos que LangChain crea
mensajes_formateados = prompt_chat.format_messages(lenguaje_programacion = "Python",
                                                   tipo_error = "IndentationError")

print("=== ESTRUCTURA DE MENSAJES PARA GEMINI ===")
for msg in mensajes_formateados:
    print(f"Rol: {msg.type.upper()} | Contenido: {msg.content}")

=== ESTRUCTURA DE MENSAJES PARA GEMINI ===
Rol: SYSTEM | Contenido: Eres un experto en Python. Tu tono es sarcástico pero educativo.
Rol: HUMAN | Contenido: ¿Por qué mi código lanza un error de tipo IndentationError?
Rol: AI | Contenido: Oh, claro, el clásico error de novato. Déjame adivinar...
Rol: HUMAN | Contenido: ¡Oye! Solo dime cómo solucionarlo, por favor.


In [7]:
# Reutilizamos nuestro prompt_chat de la celda anterior y construimos la cadena
cadena = prompt_chat | llm | parser

# Ejecutamos la cadena completa usando un diccionario de entradas
lenguaje_programacion = "Python"
tipo_error = "IndentationError"

llm_input = {"lenguaje_programacion" : lenguaje_programacion,
             "tipo_error" : tipo_error}

respuesta = cadena.invoke(input = llm_input)

print(respuesta)

Mira, no te pongas sensible. Si el intérprete de Python te está gritando "IndentationError", es porque básicamente le has entregado un bloque de código que parece escrito por un niño de tres años intentando jugar al Tetris.

Python es un lenguaje elegante, no un vertedero donde tiras líneas de código al azar. Aquí tienes la guía de supervivencia para que dejes de romper tu propio programa:

### 1. El problema: La jerarquía importa
En Python, la indentación **no es opcional**. No es como en C++ o Java, donde las llaves `{}` le dicen al compilador dónde empieza y termina algo. Aquí, el espacio en blanco es la ley. Si abres un `if`, un `for`, una función (`def`) o una clase, la siguiente línea **debe** estar más a la derecha.

### 2. La regla de oro: No mezcles
El error más común es mezclar **tabulaciones** y **espacios**.
*   Si usas tabulaciones, usa solo tabulaciones.
*   Si usas espacios, usa solo espacios (la PEP 8, nuestra biblia, recomienda **4 espacios** por nivel).

Si tu editor 

### 04. Composición Avanzada

La verdadera magia de **LCEL** y el operador `|` no es solo conectar un prompt a un modelo, sino **conectar cadenas enteras entre sí**. 

Imagina que quieres crear un flujo de trabajo donde:
1.  El modelo genera un título para un artículo basado en un tema.
2.  Ese título generado se pasa automáticamente a otro prompt para que el modelo escriba una breve sinopsis.

En código tradicional, tendrías que guardar la respuesta del modelo 1 en una variable, extraer el texto, armar un nuevo prompt y hacer una segunda llamada. Con **LCEL**, lo hacemos en una sola secuencia continua.

In [ ]:
# Definimos el modelo y el parser
llm = ChatGoogleGenerativeAI(model = MODEL, temperature = TEMPERATURE)
parser = StrOutputParser()

# Primera Cadena: Generador de Títulos

template_1 = "Escribe un título súper creativo para un artículo sobre {tema}. Solo devuelve el título, nada más."

prompt_titulo = ChatPromptTemplate.from_template(template = template_1)
cadena_titulo = prompt_titulo | llm | parser

# Segunda Cadena: Generador de Sinopsis

template_2 = "Basado en el siguiente título de un artículo: '{titulo}', escribe una sinopsis de 2 líneas atrayendo al lector. Muestra también el titulo del artículo."

prompt_sinopsis = ChatPromptTemplate.from_template(template = template_2)
cadena_sinopsis = prompt_sinopsis | llm | parser

# Unimos las cadenas, usamos un diccionario intermedio para mapear la salida de la cadena 1 a la variable de entrada de la cadena 2
cadena_completa = {"titulo": cadena_titulo} | cadena_sinopsis

# Ejecutamos el flujo completo pasando solo el "tema" inicial

tema = "viajar a marte en clase turista"

llm_input = {"tema": tema}

resultado_final = cadena_completa.invoke(input = llm_input)

print(resultado_final)

Aquí tienes la propuesta:

**Título:** Asiento 14B: Crónicas de un viaje low-cost al Planeta Rojo

**Sinopsis:**
¿Qué sucede cuando el sueño de colonizar Marte se convierte en una odisea de clase turista llena de averías y café instantáneo? Descubre la hilarante y claustrofóbica realidad de viajar al espacio con el presupuesto más ajustado de la galaxia.


### 05. Resumen:
1.  `PromptTemplate` se usa para texto simple y `ChatPromptTemplate` para estructuras de chat modernas.
2.  El operador `|` actúa como una tubería (o pipeline), pasando datos de izquierda a derecha.
3.  Podemos anidar y componer múltiples cadenas para crear flujos de trabajo complejos con muy pocas líneas de código.

In [ ]:
# Prueba con distintos modelos

from langchain_groq import ChatGroq

MODEL_GROQ = "openai/gpt-oss-120b"
TEMPERATURE = 0

# Inicializamos el cliente del modelo
llm_groq = ChatGroq(model = MODEL_GROQ, temperature = TEMPERATURE)

# Definimos el modelo y el parser
llm = ChatGoogleGenerativeAI(model = MODEL, temperature = TEMPERATURE)
parser = StrOutputParser()

# Primera Cadena: Generador de Títulos

template_1 = "Escribe un título súper creativo para un artículo sobre {tema}. Solo devuelve el título, nada más."

prompt_titulo = ChatPromptTemplate.from_template(template = template_1)
cadena_titulo = prompt_titulo | llm_groq | parser

# Segunda Cadena: Generador de Sinopsis

template_2 = "Basado en el siguiente título de un artículo: '{titulo}', escribe una sinopsis de 2 líneas atrayendo al lector. Muestra también el titulo del artículo."

prompt_sinopsis = ChatPromptTemplate.from_template(template = template_2)
cadena_sinopsis = prompt_sinopsis | llm | parser

# Unimos las cadenas, usamos un diccionario intermedio para mapear la salida de la cadena 1 a la variable de entrada de la cadena 2
cadena_completa = {"titulo": cadena_titulo} | cadena_sinopsis

# Ejecutamos el flujo completo pasando solo el "tema" inicial

tema = "viajar a marte en clase turista"

llm_input = {"tema": tema}

resultado_final = cadena_completa.invoke(input = llm_input)

print(resultado_final)

Aquí tienes la propuesta:

**Título:** ¡Marte en Clase Turista: ¡Aventuras Cósmicas sin Romper la Banca!

**Sinopsis:**
¿Quién dijo que viajar al Planeta Rojo es solo para multimillonarios? Descubre cómo la tecnología y la innovación están haciendo que el sueño de explorar Marte sea, por fin, una realidad al alcance de tu bolsillo.
